# GRPO Training — Reward-based Fine-Tuning untuk Legal Reasoning

**Modul**: Pengembangan Generative AI berbasis LLM — Digdaya Hackathon BI

**Target**: Advanced (4 pts)

---

## Ringkasan Notebook

GRPO (Group Relative Policy Optimization) menggunakan TRL `GRPOTrainer` untuk melatih model agar:

1. Menampilkan proses reasoning dalam tag `<think>...</think>`
2. Menjawab dengan Bahasa Indonesia yang baik
3. Memberikan jawaban yang akurat berdasarkan ground truth

4 Reward Functions:
1. `format_reward_func` — Reward shaping untuk format `<think>`
2. `reasoning_length_reward` — Poin proporsional berdasarkan panjang reasoning
3. `correctness_reward` — Kemiripan dengan ground truth (ROUGE-L)
4. `language_reward_func` — Penalti Inggris, bonus Indonesia

> **Catatan**: Notebook ini didesain untuk Google Colab dengan GPU T4 (16GB). Gunakan batch kecil untuk menghindari OOM.

## 1. Instalasi & Setup

In [ ]:
# Install dependencies (jika belum dari notebook sebelumnya)
%%capture
!pip install unsloth
!pip install trl transformers datasets accelerate peft bitsandbytes
!pip install rouge-score nltk huggingface_hub safetensors

# [NOTE] Jika ini notebook pertama yang dijalankan di sesi Colab baru:
# Setelah instalasi Unsloth, RESTART RUNTIME manual: Runtime → Restart runtime
# Lalu skip cell ini dan lanjut dari cell IMPORT

In [ ]:
# [PENTING] unsloth HARUS di-import paling awal sebelum transformers/trl/peft
import unsloth  # noqa: F401 — init Unsloth optimizations
import torch
import re
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from trl import GRPOTrainer, GRPOConfig
from rouge_score import rouge_scorer
import nltk

# Download NLTK data untuk ROUGE (jika belum)
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: GPU tidak terdeteksi! Pastikan Runtime → Change runtime type → T4 GPU")

In [ ]:
# Login ke HuggingFace (untuk load/push model)
from huggingface_hub import notebook_login

print("Silakan login ke HuggingFace Hub:")
notebook_login()

## 2. Load Model Hasil Fine-Tuning

Load model instruct yang sudah di fine-tune dari HuggingFace Hub (hasil dari Notebook 1).

In [ ]:
# GANTI dengan path model hasil fine-tuning Anda
HF_USERNAME = "RayhanLup1n" 
FINETUNED_MODEL = f"{HF_USERNAME}/qwen2.5-7b-indonesian-legal-finetuned"
# Jika menggunakan model lokal, gunakan path lokal
# FINETUNED_MODEL = "./qwen2.5-7b-indonesian-legal-finetuned"

max_seq_length = 2048

print(f"Loading model: {FINETUNED_MODEL}")

# Load model dengan QLoRA 4-bit (hemat VRAM untuk GRPO)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNED_MODEL,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,  # 4-bit untuk hemat VRAM saat GRPO
)

print("Model loaded.")

In [ ]:
# Setup tokenizer dengan ChatML template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Setup PEFT/LoRA untuk GRPO
# Gunakan LoRA rank rendah untuk menghemat VRAM
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank kecil untuk GRPO
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters applied untuk GRPO training.")
print(f"LoRA r=16, lora_alpha=16")

## 3. Siapkan Dataset GRPO

Dataset untuk GRPO harus berisi prompt dan ground truth answer (completion). Kita gunakan subset kecil (100-200 sample) dari dataset Alpaca Indonesia agar GRPO bisa berjalan di T4.

In [ ]:
# Load dataset Alpaca Indonesia
full_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

# Ambil subset kecil untuk GRPO (100-200 sample cukup)
GRPO_N_SAMPLES = 200
grpo_dataset = full_dataset.shuffle(seed=42).select(range(GRPO_N_SAMPLES))

print(f"GRPO dataset: {len(grpo_dataset)} samples")

In [ ]:
# Format dataset untuk GRPO: prompt + ground_truth answer
def format_grpo_sample(example):
    """
    Format sample untuk GRPO training.
    
    Dataset Ichsan2895/alpaca-gpt4-indonesian kolom:
    - input: berisi instruction/pertanyaan
    - output: berisi jawaban (ground truth)
    
    Prompt: instruksi dalam ChatML
    Ground truth: jawaban yang benar (digunakan oleh correctness_reward)
    """
    user_message = example.get("input", "") or ""
    
    # Prompt: system + user (tanpa assistant response)
    prompt = (
        f"<|im_start|>system\nAnda adalah asisten AI yang membantu menjawab pertanyaan dalam Bahasa Indonesia. "
        f"Gunakan tag <think> untuk menunjukkan proses berpikir Anda sebelum menjawab."
        f"<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    return {
        "prompt": prompt,
        "ground_truth": example.get("output", ""),
    }

grpo_dataset = grpo_dataset.map(format_grpo_sample)

# Tampilkan 1 sample
print("=" * 70)
print("CONTOH PROMPT GRPO:")
print("=" * 70)
print(grpo_dataset[0]["prompt"][:500])
print("...")
print()
print("=" * 70)
print("CONTOH GROUND TRUTH:")
print("=" * 70)
print(grpo_dataset[0]["ground_truth"][:300])
print("...")

## 4. Define Reward Functions

4 reward functions sesuai spesifikasi Advanced.

In [ ]:
def format_reward_func(completions, **kwargs):
    """
    Reward shaping untuk format <think>...</think>.
    
    Poin:
    - +0.2: Berhasil membuka dengan <think>
    - +0.3: Berhasil menutup dengan </think>
    - +1.0: Format sempurna (<think> di awal, ditutup benar, diikuti jawaban)
    - -0.5: Halusinasi (>1 tag <think> atau </think>)
    """
    rewards = []
    for completion in completions:
        # Jika completion berupa list (beberapa output per prompt), ambil yang pertama
        if isinstance(completion, list):
            completion = completion[0] if completion else ""
        
        text = completion if isinstance(completion, str) else str(completion)
        
        # Hitung jumlah tag <think> dan </think>
        think_open_count = text.count("<think>")
        think_close_count = text.count("</think>")
        
        # Penalti halusinasi: >1 tag
        if think_open_count > 1 or think_close_count > 1:
            rewards.append(-0.5)
            continue
        
        reward = 0.0
        
        # Cek apakah membuka dengan <think>
        if think_open_count == 1:
            reward += 0.2
        
        # Cek apakah menutup dengan </think>
        if think_close_count == 1:
            reward += 0.3
        
        # Cek format sempurna: <think> di awal, </think> ada, ada teks setelah </think>
        if think_open_count == 1 and think_close_count == 1:
            text_stripped = text.strip()
            # <think> harus di awal
            starts_with_think = text_stripped.startswith("<think>")
            # Cari posisi </think>
            think_close_pos = text_stripped.find("</think>")
            if think_close_pos > 7:  # Bukan tag kosong
                after_think = text_stripped[think_close_pos + 8:].strip()
                has_answer = len(after_think) > 0
                if starts_with_think and has_answer:
                    reward = 1.0  # Format sempurna
        
        rewards.append(reward)
    
    return rewards

print("format_reward_func defined.")

In [ ]:
def reasoning_length_reward(completions, **kwargs):
    """
    Poin proporsional berdasarkan jumlah karakter dalam <think>...</think>.
    
    Poin:
    - 0.0: Tidak ada tag <think> atau isinya kosong/hanya spasi
    - +0.2: Panjang < 50 karakter (terlalu singkat)
    - +0.5: Panjang 50-199 karakter
    - +1.0: Panjang >= 200 karakter (penalaran detail)
    
    Toleran terhadap reasoning yang terpotong oleh batas token.
    """
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0] if completion else ""
        
        text = completion if isinstance(completion, str) else str(completion)
        
        # Ekstrak konten di dalam tag <think>...</think>
        pattern = r"<think>(.*?)</think>"
        matches = re.findall(pattern, text, re.DOTALL)
        
        if not matches:
            # Tidak ada tag atau tag kosong
            # Cek jika ada <think> tanpa penutup (terpotong token)
            if "<think>" in text and "</think>" not in text:
                # Toleransi: ambil semua teks setelah <think> sampai akhir
                start = text.find("<think>") + 7
                content = text[start:].strip()
                if content:
                    char_count = len(content)
                    if char_count < 50:
                        rewards.append(0.2)
                    elif char_count < 200:
                        rewards.append(0.5)
                    else:
                        rewards.append(1.0)
                else:
                    rewards.append(0.0)
            else:
                rewards.append(0.0)
            continue
        
        # Ambil konten tag pertama
        content = matches[0].strip()
        char_count = len(content)
        
        if char_count == 0:
            rewards.append(0.0)
        elif char_count < 50:
            rewards.append(0.2)
        elif char_count < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)
    
    return rewards

print("reasoning_length_reward defined.")

In [ ]:
# Setup ROUGE scorer untuk correctness_reward
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def correctness_reward(completions, ground_truth, **kwargs):
    """
    Poin +1.0 jika output mengandung ground truth atau memiliki kemiripan tinggi (ROUGE-L).
    """
    rewards = []
    for completion, gt in zip(completions, ground_truth):
        if isinstance(completion, list):
            completion = completion[0] if completion else ""
        
        text = completion if isinstance(completion, str) else str(completion)
        gt_text = gt if isinstance(gt, str) else str(gt)
        
        # Ekstrak bagian setelah </think> (jawaban akhir)
        if "</think>" in text:
            answer = text.split("</think>", 1)[1].strip()
        else:
            answer = text  # Gunakan seluruh teks jika tidak ada tag
        
        # Hitung ROUGE-L F1 score antara jawaban dan ground truth
        try:
            rouge_scores = scorer.score(gt_text, answer)
            rouge_l_f1 = rouge_scores["rougeL"].fmeasure
        except Exception:
            rouge_l_f1 = 0.0
        
        # Beri reward proporsional berdasarkan ROUGE-L similarity
        # >= 0.5 similarity = +1.0; otherwise scaled
        reward = min(1.0, rouge_l_f1 / 0.5) if rouge_l_f1 > 0 else 0.0
        rewards.append(reward)
    
    return rewards

print("correctness_reward defined (ROUGE-L based).")

In [ ]:
# Daftar kata-kata Inggris umum untuk deteksi bahasa
ENGLISH_MARKERS = [
    "the", "and", "that", "have", "for", "not", "with", "you", "this", "but",
    "his", "from", "they", "say", "her", "she", "will", "one", "all", "would",
    "there", "their", "what", "about", "which", "when", "make", "like", "just",
    "into", "than", "then", "also", "because", "does", "through", "during",
]

def language_reward_func(completions, **kwargs):
    """
    -0.5 jika model menjawab dalam bahasa Inggris.
    +1.0 jika output murni Bahasa Indonesia.
    """
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0] if completion else ""
        
        text = completion if isinstance(completion, str) else str(completion)
        
        if not text.strip():
            rewards.append(0.0)
            continue
        
        # Ekstrak jawaban akhir (setelah </think>)
        if "</think>" in text:
            answer = text.split("</think>", 1)[1].strip()
        else:
            answer = text
        
        if not answer:
            rewards.append(0.0)
            continue
        
        # Hitung kata-kata Inggris dalam jawaban
        words = answer.lower().split()
        english_count = sum(1 for w in words if w in ENGLISH_MARKERS)
        
        # Jika >30% kata adalah kata Inggris umum, anggap mixed English
        english_ratio = english_count / max(len(words), 1)
        
        if english_ratio > 0.3:
            rewards.append(-0.5)
        elif english_ratio < 0.1:
            rewards.append(1.0)
        else:
            # Mixed: 0.5 (acceptable tapi tidak sempurna)
            rewards.append(0.5)
    
    return rewards

print("language_reward_func defined.")

## 5. GRPO Configuration & Training

Konfigurasi GRPO dioptimalkan untuk T4 16GB dengan QLoRA.

In [ ]:
# GRPO Configuration — optimized for T4 16GB
grpo_config = GRPOConfig(
    output_dir="./outputs/grpo",
    
    # Memory optimization — CRITICAL untuk menghindari OOM
    per_device_train_batch_size=1,  # Batch size minimum
    gradient_accumulation_steps=4,  # Efektif batch size = 4
    
    # Generation settings — [DIRECTION] num_generations & max_completion_length
    num_generations=2,  # Minimal untuk group relative (2 output per prompt)
    max_completion_length=256,  # Batasi panjang completion untuk hemat VRAM
    temperature=0.7,
    
    # Training steps
    max_steps=100,  # GRPO cukup 100-200 steps
    learning_rate=1e-5,  # Learning rate rendah untuk policy optimization
    lr_scheduler_type="cosine",
    warmup_steps=10,
    
    # Logging
    logging_steps=10,
    logging_first_step=True,
    
    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # Optimization
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.5,  # Gradient clipping lebih ketat untuk stability
    
    # Reporting
    report_to="none",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1,
    
    # GRPO-specific
    beta=0.04,  # KL penalty coefficient (default TRL)
    
    seed=42,
)

print("GRPOConfig:")
print(f"  num_generations: {grpo_config.num_generations}")
print(f"  max_completion_length: {grpo_config.max_completion_length}")
print(f"  batch_size: {grpo_config.per_device_train_batch_size}")
print(f"  max_steps: {grpo_config.max_steps}")
print(f"  learning_rate: {grpo_config.learning_rate}")

In [ ]:
# Inisialisasi GRPOTrainer
grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
)

print("GRPOTrainer initialized dengan 4 reward functions.")
print(f"Training samples: {len(grpo_dataset)}")

In [ ]:
# Jalankan GRPO training
print("=" * 60)
print("MEMULAI GRPO TRAINING")
print(f"  Reward functions: format, reasoning_length, correctness, language")
print(f"  num_generations: {grpo_config.num_generations}")
print(f"  max_completion_length: {grpo_config.max_completion_length}")
print(f"  max_steps: {grpo_config.max_steps}")
print("=" * 60)

grpo_trainer.train()

print("\nGRPO training selesai.")

## 6. Simpan & Push Model GRPO

In [ ]:
# Merge & save model hasil GRPO
GRPO_MODEL_NAME = "qwen2.5-7b-indonesian-legal-grpo"

print(f"Menyimpan model GRPO (merged_16bit)...")

model.save_pretrained_merged(
    f"./{GRPO_MODEL_NAME}",
    tokenizer,
    save_method="merged_16bit",
)

print(f"Model GRPO disimpan di: ./{GRPO_MODEL_NAME}/")

In [ ]:
# Push ke HuggingFace Hub
try:
    model.push_to_hub_merged(
        f"{HF_USERNAME}/{GRPO_MODEL_NAME}",
        tokenizer,
        save_method="merged_16bit",
        token=True,
    )
    print(f"Model GRPO berhasil diunggah: https://huggingface.co/{HF_USERNAME}/{GRPO_MODEL_NAME}")
except Exception as e:
    print(f"Push ke HF gagal (mungkin karena token tidak tersedia): {e}")
    print(f"Model tersimpan di lokal: ./{GRPO_MODEL_NAME}/")

## 7. Test Inference — Prompt Wajib

**Test Case Wajib**: 
> Prompt: "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

Output harus menampilkan proses reasoning dengan token `<think>`.

In [ ]:
def generate_response(prompt_text, max_new_tokens=512):
    """
    Generate response dari model GRPO.
    Output diharapkan mengandung <think>...</think>.
    """
    # Format sebagai ChatML
    messages = [
        {
            "from": "system",
            "value": (
                "Anda adalah asisten AI yang membantu menjawab pertanyaan hukum di Indonesia. "
                "Selalu gunakan Bahasa Indonesia yang baik dan benar. "
                "Gunakan tag <think> untuk menunjukkan proses berpikir Anda sebelum menjawab. "
                "Di dalam tag <think>, jelaskan analisis langkah demi langkah Anda. "
                "Setelah tag </think>, berikan jawaban final Anda."
            ),
        },
        {"from": "user", "value": prompt_text},
    ]
    
    # Tokenisasi dengan ChatML template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    # Generate
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    # Decode hanya token baru (setelah input)
    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=False,  # Tampilkan special tokens untuk verifikasi
    )
    
    return response

print("Fungsi generate_response() siap.")

In [ ]:
# TEST CASE WAJIB — Prompt: staf admin lembur 3 jam
MANDATORY_PROMPT = (
    "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. "
    "Apakah saya berhak dapat uang lembur?"
)

print("=" * 70)
print("TEST CASE WAJIB — GRPO Model Inference")
print("=" * 70)
print(f"Prompt: {MANDATORY_PROMPT}")
print()
print("-" * 70)
print("OUTPUT MODEL:")
print("-" * 70)

response = generate_response(MANDATORY_PROMPT)
print(response)

print()
print("-" * 70)
print("VERIFIKASI FORMAT <think>:")
print("-" * 70)
if "<think>" in response and "</think>" in response:
    print("[OK] Model menggunakan tag <think>...</think>")
    think_content = response.split("<think>", 1)[1].split("</think>", 1)[0]
    print(f"[OK] Panjang reasoning: {len(think_content)} karakter")
    answer = response.split("</think>", 1)[1].strip() if "</think>" in response else ""
    print(f"[OK] Jawaban final: {answer[:200]}..." if len(answer) > 200 else f"[OK] Jawaban final: {answer}")
else:
    print("[WARNING] Model tidak menggunakan format <think> dengan sempurna")
print("-" * 70)

## 8. Quick Test — Beberapa Prompt Tambahan

In [ ]:
# Test tambahan untuk verifikasi format
test_prompts = [
    "Apa saja hak pekerja perempuan menurut undang-undang ketenagakerjaan di Indonesia?",
    "Bagaimana prosedur pemutusan hubungan kerja yang sah secara hukum?",
    "Apa perbedaan PKWT dan PKWTT dalam hukum ketenagakerjaan Indonesia?",
]

for i, prompt in enumerate(test_prompts):
    print(f"\n{'=' * 60}")
    print(f"TEST #{i+1}: {prompt[:80]}...")
    print("=" * 60)
    
    response = generate_response(prompt, max_new_tokens=300)
    
    # Tampilkan ringkasan
    has_think = "<think>" in response
    has_close = "</think>" in response
    print(f"Format <think>: {'OK' if has_think and has_close else 'INCOMPLETE'}")
    print(f"Response length: {len(response)} chars")
    print(f"Sample: {response[:300]}..." if len(response) > 300 else response)

## Kesimpulan

Notebook ini menyelesaikan GRPO training dengan:

- [x] 4 reward functions sesuai spesifikasi
  - [x] `format_reward_func` — Reward shaping bertahap (+0.2, +0.3, +1.0, -0.5)
  - [x] `reasoning_length_reward` — Poin proporsional (0, +0.2, +0.5, +1.0)
  - [x] `correctness_reward` — ROUGE-L similarity dengan ground truth
  - [x] `language_reward_func` — Penalti Inggris (-0.5), Bonus Indonesia (+1.0)
- [x] Parameter `num_generations` dan `max_completion_length` untuk mitigasi OOM
- [x] Test case wajib: prompt staf admin lembur → output `<think>...</think>`
- [x] Model di-push ke HuggingFace Hub

Model siap digunakan di pipeline RAG pada notebook berikutnya.